# Heterochronicity parameter sweep

This example uses `perform_sweep` to test a range of heterochronicity strengths, $\lambda_h$, while holding all other model parameters fixed.

In [6]:
import torch

from gnm import defaults, evaluation, fitting
from gnm.defaults.generate_heterochronous_matrix import generate_heterochronous_matrix
from gnm.generative_rules import MatchingIndex
from gnm.utils import set_random_seed

In [7]:
seed = 42
num_iterations = 50
num_simulations = 10
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
set_random_seed(seed)

coordinates = defaults.get_coordinates(device=device)
distance_matrix = defaults.get_distance_matrix(device=device)
binary_consensus = defaults.get_binary_network(device=device)

heterochronous_matrix = generate_heterochronous_matrix(
    coord=coordinates,
    reference_coord=(0.0, 0.0, 0.0),
    sigma=torch.linalg.vector_norm(coordinates, dim=1).max().item() / 6.0,
    num_nodes=num_iterations,
    cumulative=True,
    local=False,
)

heterochronous_matrix.shape

torch.Size([50, 90, 90])

In [8]:
lambda_values = torch.linspace(0.0, 10.0, steps=11, device=device)

sweep_config = fitting.SweepConfig(
    binary_sweep_parameters=fitting.BinarySweepParameters(
        eta=torch.tensor([-2.0], device=device),
        gamma=torch.tensor([0.5], device=device),
        lambdah=lambda_values,
        distance_relationship_type=["powerlaw"],
        preferential_relationship_type=["powerlaw"],
        heterochronicity_relationship_type=["exponential"],
        generative_rule=[MatchingIndex()],
        num_iterations=[num_iterations],
    ),
    num_simulations=num_simulations,
    distance_matrix=[distance_matrix],
    heterochronous_matrix=[heterochronous_matrix],
)

lambda_values

tensor([ 0.,  1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10.])

In [9]:
energy = evaluation.MaxCriteria([
    evaluation.ClusteringKS(),
    evaluation.DegreeKS(),
    evaluation.EdgeLengthKS(distance_matrix),
])

experiments = fitting.perform_sweep(
    sweep_config=sweep_config,
    binary_evaluations=[energy],
    real_binary_matrices=binary_consensus,
    save_model=False,
    save_run_history=False,
    device=device,
    verbose=True,
)

Using device: cpu for GNM simulations


Configuration Iterations: 100%|██████████| 11/11 [00:02<00:00,  3.75it/s]

Average time per run: 0.24 seconds
Total time for sweep: 2.61 seconds


In [10]:
energy_name = str(energy)
sweep_results = [
    {
        "lambda_h": float(experiment.run_config.binary_parameters.lambdah),
        "mean_energy": experiment.evaluation_results.binary_evaluations[energy_name].mean().item(),
    }
    for experiment in experiments
]

sweep_results

[{'lambda_h': 0.0, 'mean_energy': 0.7922222018241882},
 {'lambda_h': 1.0, 'mean_energy': 0.792222261428833},
 {'lambda_h': 2.0, 'mean_energy': 0.7922222018241882},
 {'lambda_h': 3.0, 'mean_energy': 0.7844444513320923},
 {'lambda_h': 4.0, 'mean_energy': 0.7688888311386108},
 {'lambda_h': 5.0, 'mean_energy': 0.7766666412353516},
 {'lambda_h': 6.0, 'mean_energy': 0.7711111307144165},
 {'lambda_h': 7.0, 'mean_energy': 0.7744444012641907},
 {'lambda_h': 8.0, 'mean_energy': 0.7744444012641907},
 {'lambda_h': 9.0, 'mean_energy': 0.7755555510520935},
 {'lambda_h': 10.0, 'mean_energy': 0.7877777814865112}]